In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document
from llama_index.vector_stores.deeplake import DeepLakeVectorStore

c:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\deeplake\util\check_latest_version.py:32: UserWarning: A newer version of deeplake (4.5.9) is available. It's recommended that you update to the latest version using `pip install -U deeplake`.
  warnings.warn(


In [2]:
!mkdir data

A subdirectory or file data already exists.


**Pipeline 1: Collecting and preparing the documents**

In [3]:
import requests
from bs4 import BeautifulSoup
import re
import os

urls = [
    "https://github.com/VisDrone/VisDrone-dataset",
    "https://paperswithcode.com/dataset/visdrone",
    "https://openaccess.thecvf.com/content_ECCVW_2018/papers/11133/Zhu_VisDrone_Dataset_for_Object_Detection_in_ECCVW_2018_paper.pdf",
    "https://github.com/VisDrone/VisDrone2018-MOT-toolkit",
]

The corpus contains a list of sites related to CV and drones. However, the list contains also noisy links. In this case, we are working with unstructured data in various formats and variable quality.

In [4]:
def clean_text(content):
    content = re.sub(r'\[\d+\]', '', content)  # Remove reference numbers like [1], [2], etc.
    content = re.sub(r'[^\w\s]', ' ', content)  # Replace non-word characters with a single space
    return content

def fetch_and_clean(url):
    try:
        response = requests.get(url)
        response.raise_for_status()  # Check if the request was successful
        soup = BeautifulSoup(response.text, 'html.parser')
        content = soup.find('div', {'class': 'mv-parser-content'}) or soup.find('div', {'id': 'readme'})
        if content is None:
            return None
        
        for section_title in ['References', 'Acknowledgements', 'Appendix']:
            section = content.find('span', id=section_title)
            while section:
                for sib in section.parent.find_next_siblings():
                    sib.decompose()
                section.parent.decompose()
                section = content.find('span', id=section_title)
        
        text = content.get_text(separator=' ', strip=True)
        return clean_text(text)
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

Each project will reuqire specific names and paths from original data

In [5]:
output_dir = './data'
os.makedirs(output_dir, exist_ok=True)

for url in urls:
    article_name = url.split('/')[-1].replace('.html', '')
    filename = os.path.join(output_dir, f"{article_name}.txt")
    clean_article_name = fetch_and_clean(url)
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(clean_article_name or "No content available")

print("Articles fetched and cleaned successfully.")


Error fetching https://openaccess.thecvf.com/content_ECCVW_2018/papers/11133/Zhu_VisDrone_Dataset_for_Object_Detection_in_ECCVW_2018_paper.pdf: HTTPSConnectionPool(host='openaccess.thecvf.com', port=443): Max retries exceeded with url: /content_ECCVW_2018/papers/11133/Zhu_VisDrone_Dataset_for_Object_Detection_in_ECCVW_2018_paper.pdf (Caused by ConnectTimeoutError(<HTTPSConnection(host='openaccess.thecvf.com', port=443) at 0x19d92aaa3d0>, 'Connection to openaccess.thecvf.com timed out. (connect timeout=None)'))
Articles fetched and cleaned successfully.


In [6]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document
documents = SimpleDirectoryReader('./data/').load_data()
documents[0]

Document(id_='7ef970b6-8430-4855-84b7-5e193b8bc51f', embedding=None, metadata={'file_path': 'c:\\Users\\dorot\\OneDrive - Politecnico di Torino\\Desktop\\Progetti\\RAG_Books\\RAG Driven Generative AI\\Ch3\\data\\VisDrone-dataset.txt', 'file_name': 'VisDrone-dataset.txt', 'file_type': 'text/plain', 'file_size': 20, 'creation_date': '2026-03-24', 'last_modified_date': '2026-03-24'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='No content available', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}')

The SimplyDirectoryReader is designed for working with unstructured data. It recursively scans teh directory and identifies and loads all supported file types (txt, docx...). It then extracts teh content from each file and returns a list of document objects with its text and metadata, such as teh filename and filepath. 

**Pipeline 2: creating and populating a Deep lake vector store**

We will create a Deep Lake vector store and populate it with data in our documents. We will implement a standard tensor configuration with:
- text(str): the text is the content of one of the text files listed in the dictionary of documents.
- metadata(json): will contain the filename source of each chunk of text for full transparency and control.
- embedding(float32)
- id(str, auto-populated): a unique id is attributed automatically to each chunk. the vector store will also contain an index, which is  a number from 0 to n, but it cannot be used semantically since it will change each time we modify the dataset. However, the id field will remain unchanged until we decide to optimize it with index-based search strategies.

In [31]:
from llama_index.core import StorageContext

dataset_path = "./indexsearchrag"


In [32]:
vector_store = DeepLakeVectorStore(dataset_path=dataset_path, overwrite=True)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

2026-03-24 18:34:11,695 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 18:34:11,697 - INFO - Retrying request to /embeddings in 0.458481 seconds
2026-03-24 18:34:13,015 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 18:34:13,017 - INFO - Retrying request to /embeddings in 0.874073 seconds
2026-03-24 18:34:13,954 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 18:34:13,956 - INFO - Retrying request to /embeddings in 1.683132 seconds
2026-03-24 18:34:15,702 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 18:34:15,704 - INFO - Retrying request to /embeddings in 3.978775 seconds
2026-03-24 18:34:19,768 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 18:34:19,770 - INFO - Retrying request 

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

The index created will be reorganized by the indexing methods, which will rearrange and create new indexes when necessary.

In [33]:
import deeplake
ds = deeplake.load(dataset_path)

./indexsearchrag loaded successfully.



In [34]:
import json
import pandas as pd
import numpy as np

data = {}

for tensor_name in ds.tensors:
    tensor_data = ds[tensor_name].numpy()
    if tensor_data.ndim > 1:
        data[tensor_name] = [np.array(e).flatten().tolist() for e in tensor_data]
    else:
        if tensor_name == 'text':
            data[tensor_name] = [t.tobytes().decode('utf-8') for t in tensor_data]
        else:
            data[tensor_name] = tensor_data.tolist()
    
df = pd.DataFrame(data)

In [35]:
def display_record(record_number):
    record = df.iloc[record_number]
    display_data = {
        "ID": record['id'] if 'id' in record else None,
        "Text": record['text'] if 'text' in record else None,
        "Metadata": record['metadata'] if 'metadata' in record else None,
        "Embedding": record['embedding'] if 'embedding' in record else None,
    }

In [36]:
rec = 0
display_record(rec)

IndexError: single positional indexer is out-of-bounds

The id is the index we will be using to organize teh chunks of text of teh text colum in the dataset. The chunks will be transformed into nodes that can contain teh original text, summaries and additional information. 

The metadata was automatically generated by SimpleDirectoryReader.

The text column contains the chunked data.

The embedding of each chunk of datawas generated through an embedding model that we do not need to configure.

**Pipeline 3: Index-based RAG**

We will implement four index engines:
- Vector Store index Engine: creates a vector store index from the documents, enabling efficient similarity-based searches.
- Tree Index: builds a hierarchical tree index from teh documents.
- List Index: constructs a straightfroward idnex from the documents.
- Keyword table Index: creates an index based on keywords extracted from the documents.

We will implementing querying with an LLM: queries the index with user input, retrieves teh relevanmt documents and returns a synthesized response.

In [ ]:
user_input = "What is the VisDrone dataset?"

In [38]:
k = 3 #number of retrieved documents to use for the generation of the answer
temp = 0.1 #temperature for the generation of the answer
mt = 1024 #number of tokens to use for the retrieved context when generating the answer

In [41]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def calculate_similarity(text1, text2):
    emb1 = model.encode([text1])
    emb2 = model.encode([text2])
    similarity = cosine_similarity([emb1], [emb2])
    return similarity[0][0]

2026-03-24 18:59:05,598 - INFO - Use pytorch device_name: cuda:0
2026-03-24 18:59:05,599 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


1) Vector Store Index Query Engine
VectorStoreIndex is a type of index within LlamaIndex that implements vector embeddings to represent and retrieve infromation from docs. These docs with similar meanings will have embeddings that are closer together in teh vector space. However, this time, it does not use the existing Deep Lake vector store, it can create a new memory vector index, re-embed the docs and create a new index structure. 

In [45]:
from llama_index.core import VectorStoreIndex
vector_store_index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
print(type(vector_store_index))

2026-03-24 19:05:05,379 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:05,381 - INFO - Retrying request to /embeddings in 0.401726 seconds
2026-03-24 19:05:05,870 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:05,870 - INFO - Retrying request to /embeddings in 0.855444 seconds
2026-03-24 19:05:06,813 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:06,815 - INFO - Retrying request to /embeddings in 1.555540 seconds
2026-03-24 19:05:08,468 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:08,469 - INFO - Retrying request to /embeddings in 3.368408 seconds
2026-03-24 19:05:11,893 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:11,895 - INFO - Retrying request 

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [44]:
vector_query_engine = vector_store_index.as_query_engine(similarity_top_k=k, temperature=temp, max_tokens=mt)

NameError: name 'vector_store_index' is not defined

This function executes a query using a vector query engine and processes teh results into a structured format.

In [ ]:
import pandas as pd
import textwrap

def index_query(input_query):
    response = vector_query_engine.query(input_query)
    print(textwrap.fill(str(response), width=80))
    node_data =[]
    for node in response.source_nodes:
        noe_info = {
            "id": node.node_id,
            "text": node.node.get_text(),
            "score" : node.score
        }
        node_data.append(noe_info)
    df = pd.DataFrame(node_data)
    return df, response

Optimized chunking: we can decide the chunk size or teh code determines teh chunk size automatically.

In [ ]:
for node in response.source_nodes:
    node = node.node
    chunk_size = len(node.get_text())
    print(f"Node ID: {node.node_id}, Chunk Size: {chunk_size}")


We first calculate the sum of the scores and the average scores, and then we divide the weighted average by the time elapsed to perform the query. This metric is not an absolute value. It's an indicator that we can use to compare this engine with others.



In [ ]:
import numpy as np

def info_metrics(response):
    scores = [node.score for node in response.source_nodes]
    if scores:
        weights = np.exp(scores) / np.sum(np.exp(scores))  # Softmax to get weights
        perf = np.average(scores, weights=weights) / elapsed_time  # Example performance metric 
    else:
        perf = 0
    return perf

2) Tree index query Engine
The tree index organizes docs in a tree structure with broader summaries at higher levels and detailed information lower levels. Each node in the tree summarizes the text it covers. It is efficient for large datasets and queries large collections of docs rapisly by breaking them down into optimized chunks. In this model, the LLM acts like it is answering a multiple-choice question when selecting the best nodes during a query. It analyzes the query, compares it with teh summaries of the current node's children and decides which path to follow to find the most relevant information.

In [46]:
from llama_index.core import TreeIndex
tree_index = TreeIndex.from_documents(documents)   

In [49]:
import time
import textwrap

start_time = time.time()
response = tree_query_engine.query(user_input)
end_time = time.time()
elapsed_time = end_time - start_time

NameError: name 'tree_query_engine' is not defined

In [ ]:
similarity_score = calculate_similarity(user_input, response.get_text())
performance = similarity_score / elapsed_time if elapsed_time > 0 else 0

3) Last index query engine
The query engine will process the user input and each doc as a prompt for an LLM. The LLm will evaluate the semantic similarity relationship between the docs and the query, thus implicity ranking and selecting teh most relavnt nodes. The selection with an LLM is not rule-based. Nothing is predefined, which means that the selction is prompt-based by combining the user input with a collection of docs. The LLM evaluates each doc in teh list indipendently, assigning a score based on its perceived relevance to teh query. Then, teh top-k docs are retained by the query engine.

In [ ]:
from llam_index.core import ListIndex
list_index = ListIndex.from_documents(documents)

In [ ]:
list_query_engine = list_index.as_query_engine(similarity_top_k=k, temperature=temp, max_tokens=mt)

In [ ]:
start_time = time.time()
response = list_query_engine.query(user_input)
end_time = time.time()
elapsed_time = end_time - start_time

In [ ]:
similarity_score = calculate_similarity(user_input, response.get_text())
performance = similarity_score / elapsed_time if elapsed_time > 0 else 0

4) Keyword index query engine
It is designed to extract keywords from your docs and organize them in a table-like structure. This strcuture makes it easier to query and retrieve relevant information based on specific keyowrds or topics.

In [50]:
from llama_index.core import KeywordTableIndex  
keyword_index = KeywordTableIndex.from_documents(documents)

2026-03-24 19:52:15,799 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:15,803 - INFO - Retrying request to /chat/completions in 0.444031 seconds
2026-03-24 19:52:16,986 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:16,988 - INFO - Retrying request to /chat/completions in 0.867025 seconds
2026-03-24 19:52:18,546 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:18,548 - INFO - Retrying request to /chat/completions in 1.589668 seconds
2026-03-24 19:52:20,861 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:20,866 - WARNING - Retrying llama_index.llms.openai.base.OpenAI._chat in 1 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
data = []
for keyword, doc_ids in keyword_index.keyword_to_doc_ids.items():
    for doc_id in doc_ids:
        data.append({"keyword": keyword, "doc_id": doc_id})

df = pd.DataFrame(data)
df